<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания № 8


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[Создать базовый класс CreditCard в C#, который будет представлять информацию о кредитных картах. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [7]:
using System;
using System.Collections.Generic;

public delegate void TransactionHandler(string message);

public class CreditCard
{
    public event TransactionHandler OnPayment;
    public event TransactionHandler OnDeposit;

    public string CardNumber { get; set; }
    public string HolderName { get; set; }
    public string ExpiryDate { get; set; }
    protected decimal Balance { get; set; }
    public List<string> Transactions { get; set; }
    public decimal DailyLimit { get; set; }
    public bool IsBlocked { get; set; }

    public CreditCard(string cardNumber, string holderName, string expiryDate, decimal balance = 0)
    {
        CardNumber = cardNumber;
        HolderName = holderName;
        ExpiryDate = expiryDate;
        Balance = balance;
        Transactions = new List<string>();
        DailyLimit = 20000;
        IsBlocked = false;
    }

    public void Deposit(decimal amount)
    {
        if (IsBlocked) return;
        Balance += amount;
        Transactions.Add($"Пополнение: +{amount:C}");
        OnDeposit?.Invoke($"Карта {CardNumber}: пополнение на {amount:C}");
    }

    public virtual string GetInfo()
    {
        return $"Карта: {CardNumber}\nВладелец: {HolderName}\nСрок действия: {ExpiryDate}";
    }

    public virtual string Pay(decimal amount)
    {
        if (IsBlocked) return "Карта заблокирована.";
        if (amount > DailyLimit) return $"Превышен дневной лимит ({DailyLimit:C}).";
        if (Balance >= amount)
        {
            Balance -= amount;
            Transactions.Add($"Покупка: -{amount:C}");
            OnPayment?.Invoke($"Оплата {amount:C} с карты {CardNumber}");
            return $"Оплата на сумму {amount:C} прошла успешно!";
        }
        return $"Недостаточно средств! Баланс: {Balance:C}";
    }

    public virtual string CheckBalance()
    {
        return $"Баланс карты: {Balance:C}";
    }

    public void BlockCard()
    {
        IsBlocked = true;
    }

    public void PrintTransactions()
    {
        Console.WriteLine($"История операций {CardNumber}:");
        foreach (var t in Transactions) Console.WriteLine("  " + t);
    }
}

public class GoldCreditCard : CreditCard
{
    public int BonusMiles { get; set; }
    public double CashbackRate { get; set; }
    public int StatusLevel { get; set; }
    public List<string> Bonuses { get; set; }

    public GoldCreditCard(string cardNumber, string holderName, string expiryDate, decimal balance = 0, int bonusMiles = 0)
        : base(cardNumber, holderName, expiryDate, balance)
    {
        BonusMiles = bonusMiles;
        CashbackRate = 0.02;
        StatusLevel = 1;
        Bonuses = new List<string>();
    }

    public override string Pay(decimal amount)
    {
        var result = base.Pay(amount);
        if (!result.Contains("успешно")) return result;

        int miles = (int)(amount / 100);
        BonusMiles += miles;

        decimal cashback = amount * (decimal)CashbackRate;
        Deposit(cashback);

        Bonuses.Add($"Кэшбэк {cashback:C}");

        return result + $" Миль начислено: {miles}. Кэшбэк: {cashback:C}.";
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nТип: Gold\nБонусные мили: {BonusMiles}\nСтатус: {StatusLevel}";
    }

    public void IncreaseStatus()
    {
        StatusLevel++;
    }
}

public class PremiumCreditCard : CreditCard
{
    public string SupportAssistant { get; set; }
    public decimal CreditLimit { get; set; }
    public decimal MonthlyFee { get; set; }
    public List<string> VIPServices { get; set; }

    public PremiumCreditCard(string cardNumber, string holderName, string expiryDate, string supportAssistant, decimal balance = 0)
        : base(cardNumber, holderName, expiryDate, balance)
    {
        SupportAssistant = supportAssistant;
        CreditLimit = 50000;
        MonthlyFee = 500;
        VIPServices = new List<string>() { "Аренда авто", "Страхование" };
    }

    public override string CheckBalance()
    {
        if (Balance < 1000) return $"Баланс карты: {Balance:C}\nВнимание! Низкий баланс. Обратитесь: {SupportAssistant}";
        return base.CheckBalance();
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nТип: Premium\nАссистент: {SupportAssistant}\nЛимит: {CreditLimit:C}";
    }

    public void AddVIPService(string service)
    {
        VIPServices.Add(service);
    }
}

public class CorporateCreditCard : CreditCard
{
    public string Company { get; set; }
    public int EmployeeCount { get; set; }
    public decimal MonthlyLimit { get; set; }
    public List<string> EmployeeCards { get; set; }

    public CorporateCreditCard(string cardNumber, string holderName, string expiryDate, string company, decimal balance = 0)
        : base(cardNumber, holderName, expiryDate, balance)
    {
        Company = company;
        EmployeeCount = 0;
        MonthlyLimit = 100000;
        EmployeeCards = new List<string>();
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nТип: Corporate\nКомпания: {Company}";
    }

    public void AddEmployeeCard(string name)
    {
        EmployeeCards.Add(name);
        EmployeeCount++;
    }
}

Console.WriteLine("=== Демонстрация работы с кредитными картами ===\n");

CreditCard basicCard = new CreditCard("1234 5678 9012 3456", "Иван Иванов", "12/25", 5000);
GoldCreditCard goldCard = new GoldCreditCard("2345 6789 0123 4567", "Петр Петров", "06/26", 10000, 150);
PremiumCreditCard premiumCard = new PremiumCreditCard("3456 7890 1234 5678", "Мария Сидорова", "09/27", "support@premiumbank.ru", 800);
CorporateCreditCard corporateCard = new CorporateCreditCard("4567 8901 2345 6789", "Алексей Козлов", "03/28", "ООО 'Технологии'", 20000);

basicCard.OnPayment += msg => Console.WriteLine("[Событие] " + msg);
goldCard.OnPayment += msg => Console.WriteLine("[Событие] " + msg);
goldCard.OnDeposit += msg => Console.WriteLine("[Событие] " + msg);

Console.WriteLine("=== Базовая карта ===");
Console.WriteLine(basicCard.Pay(3000));
Console.WriteLine();

Console.WriteLine("=== Gold карта ===");
Console.WriteLine(goldCard.Pay(2500));
Console.WriteLine();

Console.WriteLine("=== Premium карта ===");
Console.WriteLine(premiumCard.CheckBalance());
Console.WriteLine();

Console.WriteLine("=== Corporate карта ===");
corporateCard.AddEmployeeCard("Сотрудник №1");
Console.WriteLine(corporateCard.GetInfo());
Console.WriteLine();

Console.WriteLine("=== Полиморфизм ===");
CreditCard[] cards = { basicCard, goldCard, premiumCard, corporateCard };
foreach (var card in cards)
{
    Console.WriteLine(card.GetInfo());
    Console.WriteLine(card.Pay(1000));
    Console.WriteLine(card.CheckBalance());
    Console.WriteLine("---");
}


=== Демонстрация работы с кредитными картами ===

=== Базовая карта ===
[Событие] Оплата ¤3,000.00 с карты 1234 5678 9012 3456
Оплата на сумму ¤3,000.00 прошла успешно!

=== Gold карта ===
[Событие] Оплата ¤2,500.00 с карты 2345 6789 0123 4567
[Событие] Карта 2345 6789 0123 4567: пополнение на ¤50.00
Оплата на сумму ¤2,500.00 прошла успешно! Миль начислено: 25. Кэшбэк: ¤50.00.

=== Premium карта ===
Баланс карты: ¤800.00
Внимание! Низкий баланс. Обратитесь: support@premiumbank.ru

=== Corporate карта ===
Карта: 4567 8901 2345 6789
Владелец: Алексей Козлов
Срок действия: 03/28
Тип: Corporate
Компания: ООО 'Технологии'

=== Полиморфизм ===
Карта: 1234 5678 9012 3456
Владелец: Иван Иванов
Срок действия: 12/25
[Событие] Оплата ¤1,000.00 с карты 1234 5678 9012 3456
Оплата на сумму ¤1,000.00 прошла успешно!
Баланс карты: ¤1,000.00
---
Карта: 2345 6789 0123 4567
Владелец: Петр Петров
Срок действия: 06/26
Тип: Gold
Бонусные мили: 175
Статус: 1
[Событие] Оплата ¤1,000.00 с карты 2345 6789 0123 